#### Feature Engineering
Here I'll create new features from the cleaned data and prepare a model-ready dataset for regression and classification.

#### Step-0: Setup

In [1]:
# Imports and Load
import numpy as np
import pandas as pd

df = pd.read_csv("../data/cleaned/charging_clean.csv", parse_dates=['StartDatetime', 'EndDatetime'])
df.shape

(72088, 9)

#### Step-1: Sorting 

In [2]:
# Sorting sessions from oldest to newest
df = df.sort_values('StartDatetime').reset_index(drop=True)
df.head()

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDatetime,EndDatetime,Duration,Demand
0,102,291,1,public institution,0,2021-09-30 17:14:00,2021-10-01 01:21:00,487,27.68
1,104,191,1,resort,0,2021-09-30 18:22:00,2021-10-01 02:21:00,478,55.13
2,105,320,1,camping,0,2021-09-30 18:57:00,2021-10-01 00:44:00,347,38.45
3,0,433,1,camping,0,2021-09-30 19:52:00,2021-10-01 00:21:00,269,30.17
4,391,250,1,public area,0,2021-09-30 20:48:00,2021-10-01 03:35:00,406,47.00


In [3]:
df.tail()

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDatetime,EndDatetime,Duration,Demand
72083,0,126,1,resort,0,2022-09-30 21:49:00,2022-09-30 23:34:00,105,11.70
72084,69,967,0,public parking lot,1,2022-09-30 22:10:00,2022-09-30 22:40:00,30,15.79
72085,0,219,1,public area,0,2022-09-30 22:31:00,2022-09-30 23:25:00,55,5.83
72086,531,66,1,resort,0,2022-09-30 22:37:00,2022-09-30 23:52:00,75,8.49
72087,0,118,1,public institution,0,2022-09-30 23:11:00,2022-09-30 23:13:00,2,0.10


#### Step-2: Time features

In [4]:
# Creating time features from start time
df['Hour']= df['StartDatetime'].dt.hour
df['Weekday']= df['StartDatetime'].dt.dayofweek
df['IsWeekend']= (df['Weekday']>=5).astype(int)
df[['StartDatetime', 'Hour', 'Weekday', 'IsWeekend']].head()

,StartDatetime,Hour,Weekday,IsWeekend
0,2021-09-30 17:14:00,17,3,0
1,2021-09-30 18:22:00,18,3,0
2,2021-09-30 18:57:00,18,3,0
3,2021-09-30 19:52:00,19,3,0
4,2021-09-30 20:48:00,20,3,0


#### Step-3: Guest flag

In [5]:
# 1 if the session is from UserID 0 (likely unregistered users), else 0
df['IsGuest']= (df['UserID']== 0).astype(int)
df['IsGuest'].value_counts()

IsGuest
0    40878
1    31210
Name: count, dtype: int64

#### Step-4: User History Features

In [6]:
# Number of earlier sessions of the same user
df['UserPastSessions']= df.groupby('UserID').cumcount()

In [7]:
# Total demand of earlier sessions, then the average
df['UserPastTotal']= df.groupby('UserID')['Demand'].cumsum()-df['Demand']
df['UserPastAvgDemand']= df['UserPastTotal']/df['UserPastSessions']

In [8]:
# Guests are many different people, so they have no personal history
df.loc[df['IsGuest']== 1, 'UserPastSessions']= 0
df.loc[df['IsGuest']== 1, 'UserPastAvgDemand']= np.nan

# UserPastTotal was only a helper column
df= df.drop(columns='UserPastTotal')

In [9]:
# Checking one user (UserID 31, the most active user from Q10)
df[df['UserID']==31][['StartDatetime', 'Demand', 'UserPastSessions', 'UserPastAvgDemand']].head()

,StartDatetime,Demand,UserPastSessions,UserPastAvgDemand
9645,2021-12-10 17:13:00,4.9,0,NaN
9679,2021-12-10 19:08:00,1.9,1,4.900000
10048,2021-12-13 13:25:00,8.9,2,3.400000
10377,2021-12-15 13:20:00,5.9,3,5.233333
10511,2021-12-16 10:54:00,10.8,4,5.400000


#### Step-5: Charger history features

In [10]:
# Number of earlier sessions at the same charger
df['ChargerPastSessions']= df.groupby('ChargerID').cumcount()

# Average demand of earlier sessions at the same charger
df['ChargerPastTotal']= df.groupby('ChargerID')['Demand'].cumsum()-df['Demand']
df['ChargerPastAvgDemand']= df['ChargerPastTotal']/df['ChargerPastSessions']

# Helper column not needed
df= df.drop(columns='ChargerPastTotal')

In [11]:
# Checking one charger (ChargerID 573, the busiest charger from Q10)
df[df['ChargerID']==573][['StartDatetime', 'Demand', 'ChargerPastSessions', 'ChargerPastAvgDemand']].head()

,StartDatetime,Demand,ChargerPastSessions,ChargerPastAvgDemand
127,2021-10-01 18:58:00,6.36,0,NaN
187,2021-10-02 11:32:00,8.57,1,6.360000
330,2021-10-03 14:05:00,7.43,2,7.465000
355,2021-10-03 17:40:00,4.38,3,7.453333
363,2021-10-03 18:57:00,9.52,4,6.685000


#### Step-6: Dropping columns not needed for modelling
1. UserID and ChargerID are only labels; their information is now in IsGuest and the history features.
2. EndDatetime is only known after a session ends, so it would leak the answer.
3. StartDatetime is kept only for the time-based split in 06 (not used as a feature).
4. Duration is kept only for the comparison model (Model B) in 06.

In [12]:
# Dropping ID and end time columns
df= df.drop(columns=['UserID', 'ChargerID', 'EndDatetime'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72088 entries, 0 to 72087
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   ChargerCompany        72088 non-null  int64         
 1   Location              72088 non-null  object        
 2   ChargerType           72088 non-null  int64         
 3   StartDatetime         72088 non-null  datetime64[ns]
 4   Duration              72088 non-null  int64         
 5   Demand                72088 non-null  float64       
 6   Hour                  72088 non-null  int32         
 7   Weekday               72088 non-null  int32         
 8   IsWeekend             72088 non-null  int64         
 9   IsGuest               72088 non-null  int64         
 10  UserPastSessions      72088 non-null  int64         
 11  UserPastAvgDemand     38558 non-null  float64       
 12  ChargerPastSessions   72088 non-null  int64         
 13  ChargerPastAvgDe